# Error agreement with SLAE-CS across encoders (PANAV test set, 10-seed ensembles)

Pearson and Spearman correlation of each model's per-residue |error| with that of SLAE-CS, on the residues shared by all models. SLAE-CS ceilings: one seed vs the other nine, and 5 vs 5 seeds.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, rankdata


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })


set_style(16)
BLUE, RED, GREY = "#61AFEF", "#E06C75", "#999999"

RES = "../../data/analysis/Fig4/results"
GT = "../../data/rBMRB/cluster_30p/PANAV/test.json"
FIG = "Figs"

REFERENCE = ("SLAE-CS", "predictions_N_seed_ensemble_10seed.json")
MODELS = [  # (label, file) -- fixed order, grouped by encoder input
    ("ESM3\n(str + seq)", "predictions_N_seed_ensemble_10seed_esm3.json"),
    ("MACE-OFF23", "predictions_N_seed_ensemble_10seed_mace.json"),
    ("ESM3\n(seq only)", "predictions_N_seed_ensemble_10seed_esm3_seqonly.json"),
    ("ProteinMPNN\n(fine-tuned)", "predictions_N_seed_ensemble_10seed_mpnn_ft.json"),
]
SHOW_CEILING = True   # grey SLAE-CS internal reference bars

In [ ]:
# ---------- per-residue |error| for every model on the shared residue set ----------
gt = json.load(open(GT))
key_of = {v: k for k, v in gt["entry_ID"].items()}


def residue_table(pred_file, per_seed=False):
    """(entry, resnum) -> |mean - exp| for one ensemble; optionally also each seed's |pred - exp|."""
    d = json.load(open(f"{RES}/{pred_file}"))
    rows = []
    for entry, key in key_of.items():
        if entry not in d["mean"]:
            continue
        seq, shifts = gt["sequence"][key], gt["N"][key]
        n = len(seq)
        if len(d["mean"][entry]) < n or len(d["mask"][entry]) < n:   # right-padded; shorter = unusable
            continue
        mean = np.array(d["mean"][entry], dtype=float)[:n]
        mask = np.array(d["mask"][entry], dtype=bool)[:n]
        if per_seed:
            seeds = np.array(d["pred"][entry], dtype=float)[:, :n]    # (n_seeds, n)
        for i in range(n):
            if seq[i] == "P" or shifts[i] is None or not mask[i]:
                continue
            row = dict(entry=entry, resnum=i + 1, err=abs(mean[i] - shifts[i]))
            if per_seed:
                row["seed_pred"] = seeds[:, i]
                row["exp"] = float(shifts[i])
            rows.append(row)
    return pd.DataFrame(rows).set_index(["entry", "resnum"])


ref = residue_table(REFERENCE[1], per_seed=True)
data = ref.rename(columns={"err": REFERENCE[0]})
for label, pred_file in MODELS:
    data = data.join(residue_table(pred_file)["err"].rename(label), how="inner")   # intersection of masks
data = data.reset_index()

SEED_PRED = np.stack(data["seed_pred"].to_numpy())          # (n_res, n_seeds)
EXP = data["exp"].to_numpy()
N_SEEDS = SEED_PRED.shape[1]
assert np.allclose(np.abs(SEED_PRED.mean(1) - EXP), data[REFERENCE[0]].to_numpy()), "seed average must equal `mean`"

print(f"shared residues: {len(data)} | proteins: {data.entry.nunique()} | SLAE-CS seeds: {N_SEEDS}")
print("median |error| (ppm):", {c.replace(chr(10), ' '): round(data[c].median(), 3)
                                for c in [REFERENCE[0]] + [m for m, _ in MODELS]})

In [ ]:
# ---------- Pearson / Spearman of |error| ----------
def abs_err(pred):
    return np.abs(pred - EXP)


def both(a, b):
    return pearsonr(a, b)[0], spearmanr(a, b)[0]


rows = []
if SHOW_CEILING:
    # seed k vs mean of the other 9 (leave-one-out), averaged over the 10 seeds
    loo = np.array([both(abs_err(SEED_PRED[:, k]), abs_err(np.delete(SEED_PRED, k, axis=1).mean(1)))
                    for k in range(N_SEEDS)])
    rows.append(dict(model="SLAE-CS\n(inter-seed variability)", pearson=loo[:, 0].mean(), spearman=loo[:, 1].mean()))
    print(f"seed vs rest: Pearson {loo[:, 0].min():.3f}-{loo[:, 0].max():.3f}, "
          f"Spearman {loo[:, 1].min():.3f}-{loo[:, 1].max():.3f} across the {N_SEEDS} seeds")

ref_err = data[REFERENCE[0]].to_numpy()
for label, _ in MODELS:
    p, s = both(ref_err, data[label].to_numpy())
    rows.append(dict(model=label, pearson=p, spearman=s))

C = pd.DataFrame(rows)
N_REF = 1 if SHOW_CEILING else 0
print()
print(C.assign(model=C.model.str.replace("\n", " ")).round(3).to_string(index=False))

In [ ]:
# ---------- figure: grouped bars, Pearson vs Spearman ----------
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(C))
w = 0.4

for offset, metric, color in [(-w / 2, "pearson", "#3F729B"), (w / 2, "spearman", "#A9CCE3")]:
    vals = C[metric].to_numpy()
    colors = [color for i in range(len(C))]     # SLAE-CS ceilings in grey
    bars = ax.bar(x + offset, vals, w, color=colors, edgecolor="white", linewidth=1.5)
    # for b in bars[:N_REF]:
    #     b.set_hatch("//" if metric == "pearson" else "..")          # hatch keeps the metric readable on grey
    for xi, v in zip(x + offset, vals):
        ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=16, font="Arial")

if N_REF:
    ax.axvline(N_REF - 0.5, color="#BBBBBB", lw=1, ls="--")
ax.set_xticks(x)
ax.set_xticklabels(C.model)
ax.set_ylabel("correlation with SLAE-CS |error|")
ax.set_ylim(0, 1.1)
ax.legend(handles=[Patch(color="#3F729B", label="Pearson r"), Patch(color="#A9CCE3", label="Spearman ρ")],
          frameon=False, loc="upper right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
fig.savefig(f"{FIG}/error_corr_vs_slaecs_panav10seedens.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/error_corr_vs_slaecs_panav10seedens.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved error_corr_vs_slaecs_panav10seedens.svg/.png")

In [ ]:
# ---------- heatmap: pairwise Pearson r of per-residue |error| between all models ----------
# Same shared residue set as the bars above. The matrix is symmetric with a trivial diagonal, so only the
# strictly lower triangle is drawn (rows = models 2..n, columns = models 1..n-1).
set_style(19)
ORDER = [REFERENCE[0]] + [label for label, _ in MODELS]
R = data[ORDER].corr(method="pearson")
print(R.rename(index=lambda s: s.replace("\n", " "), columns=lambda s: s.replace("\n", " ")).round(3).to_string())

rows_lab, cols_lab = ORDER[1:], ORDER[:-1]
M = R.loc[rows_lab, cols_lab].to_numpy().copy()
M[np.triu_indices(len(rows_lab), k=1)] = np.nan          # keep i > j only

set_style(16)
fig, ax = plt.subplots(figsize=(7.2, 6.0))
im = ax.imshow(M, cmap="Blues", vmin=0.4, vmax=1.0)
for i in range(M.shape[0]):
    for j in range(i + 1):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=20, font="Arial",
                color="white" if M[i, j] > 0.75 else "black")
ax.set_xticks(range(len(cols_lab))); ax.set_xticklabels(cols_lab, fontsize=16, font="Arial")
ax.set_yticks(range(len(rows_lab))); ax.set_yticklabels(rows_lab, fontsize=16, font="Arial")
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)

cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Pearson r of per-residue |error|", fontsize=18, font="Arial")
cb.ax.tick_params(labelsize=18)                       # tick label font size

plt.tight_layout()
fig.savefig(f"{FIG}/error_corr_heatmap_pearson_panav10seedens.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/error_corr_heatmap_pearson_panav10seedens.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved error_corr_heatmap_pearson_panav10seedens.svg/.png")
if SHOW_CEILING:
    print(f"(for the caption) SLAE-CS inter-seed ceiling, seed vs rest: r = {C.pearson.iloc[0]:.2f}")


In [ ]:
# ---------- averaged per-protein Kendall's tau, prediction vs experiment ----------
# Same recipe as Fig2/kendalltau_comparison_results_10seedens.ipynb (GT parsing, per-protein masks, MIN_VALID_RES,
# NaN tau -> 0, mean +/- std over proteins). GT file is identical to Fig2's test_panav.json (same md5).
import math
from pathlib import Path
from scipy.stats import kendalltau


def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    """entry_ID (bmr prefix stripped) -> (shift list, mask list)."""
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null")) for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        entry = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[entry] = (shift, mask)
    return out


def build_per_protein(pred_path, gt_path, atom):
    """entry -> {pred, gt}: GT mask (None / Pro-N) intersected with the prediction's own mask."""
    with open(pred_path) as f:
        d = json.load(f)
    pred_dict, mask_dict = d["mean"], d["mask"]
    gt_map = load_ground_truth(Path(gt_path), atom=atom)
    per_protein, skipped = {}, []
    for entry, (true_shift, true_mask) in gt_map.items():
        if entry not in pred_dict:
            skipped.append(entry)
            continue
        pred_shift, pred_mask = pred_dict[entry], mask_dict[entry]
        n = min(len(true_shift), len(true_mask), len(pred_shift), len(pred_mask))
        combined_mask = np.array(true_mask[:n], dtype=bool) & np.array(pred_mask[:n], dtype=bool)
        if not combined_mask.any():
            skipped.append(entry)
            continue
        per_protein[entry] = {
            "pred": np.asarray(pred_shift[:n], dtype=float)[combined_mask],
            "gt": np.asarray(true_shift[:n], dtype=float)[combined_mask],
        }
    return per_protein, skipped


ATOM = "N"
MIN_VALID_RES = 3
ALL_MODELS = [REFERENCE] + MODELS
per_protein_by_model, skipped_by_model = {}, {}
for label, pred_file in ALL_MODELS:
    per_protein_by_model[label], skipped_by_model[label] = build_per_protein(f"{RES}/{pred_file}", GT, ATOM)

common_proteins = set.intersection(*(set(v) for v in per_protein_by_model.values()))
common_proteins = sorted(common_proteins)

kendall_by_model = {label: [] for label, _ in ALL_MODELS}
skipped_short = []
for entry in common_proteins:
    if len(per_protein_by_model[REFERENCE[0]][entry]["gt"]) < MIN_VALID_RES:
        skipped_short.append(entry)
        continue
    for label, _ in ALL_MODELS:
        pp = per_protein_by_model[label][entry]
        tau = kendalltau(pp["pred"], pp["gt"])[0]
        kendall_by_model[label].append(0.0 if np.isnan(tau) else tau)

K = pd.DataFrame([dict(model=label, tau=np.mean(kendall_by_model[label]), std=np.std(kendall_by_model[label]),
                       n_proteins=len(kendall_by_model[label])) for label, _ in ALL_MODELS])
print(K.assign(model=K.model.str.replace("\n", " ")).round(4).to_string(index=False))
print(f"\nSkipped (missing from a model's predictions): { {l.replace(chr(10), ' '): len(s) for l, s in skipped_by_model.items()} }")
print(f"Skipped (< {MIN_VALID_RES} valid residues): {len(skipped_short)}: {skipped_short}")

# ---------- figure ----------
fig, ax = plt.subplots(figsize=(7.2, 4))
x = np.arange(len(K))
ax.bar(x, K["tau"], 0.8, yerr=K["std"], capsize=4, color=BLUE, edgecolor="white", linewidth=1.5,
       error_kw=dict(elinewidth=1.2, ecolor="gray", capthick=1.2))
for xi, v, s in zip(x, K["tau"], K["std"]):
    # ax.text(xi, v + s + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=16, font="Arial")
    print(f"{v:.2f}")
ax.set_xticks(x)
ax.set_xticklabels(K.model)
for tick, label in zip(ax.get_xticklabels(), K.model):
    if label == REFERENCE[0]:            # "SLAE-CS"
        tick.set_fontweight("bold")
ax.set_ylabel("Kendall's τ (mean ± std)")
ax.set_ylim(0, 1.0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
fig.savefig(f"{FIG}/pred_kendall_vs_exp_panav10seedens.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/pred_kendall_vs_exp_panav10seedens.png", dpi=300, bbox_inches="tight")
plt.show()